# Qwen2.5-1.5B 예비 양자화 실험

이 노트북은 교수님 코멘트에 따라 최종 주제를 1번과 3번 중에서 고르기 위한 **예비 실험**입니다.

비교군은 FP16, RTN 4bit, 영어 위키 보정 GPTQ 4bit, 한국어 위키 보정 GPTQ 4bit입니다. 한국어와 영어 평가 문서를 각각 세 묶음으로 나누고, 각 언어에서 FP16 대비 Perplexity 저하율을 계산합니다. 원점수 자체는 언어 간에 직접 비교하지 않습니다.

예비 실험에서 확인할 내용:

1. 4bit 양자화의 한국어 저하율과 영어 저하율이 각 조건 및 평가 묶음에서 어떤 양상을 보이는가?
2. 영어 보정과 한국어 보정 GPTQ가 한국어 평가에서 차이를 만드는가?
3. 결과가 세 평가 묶음에서 일관적인가, 특정 묶음에만 나타나는가?

결과를 본 뒤에 1번(언어별 저하 분석)과 3번(한국어 품질 개선 실험) 중 하나를 정합니다. 이 노트북은 그 선택 기준을 미리 고정하지 않습니다.

**주의:** WikiText-2 영어 test와 한국어 Wikipedia는 서로 다른 평가 말뭉치입니다. 따라서 영어와 한국어의 절대 Perplexity를 비교하지 않고, 각 말뭉치 안에서 FP16 대비 상대 변화만 비교합니다. Instruct 모델에 일반 문서를 넣어 계산한 PPL은 다음 토큰 예측의 비교 지표이며, 대화 응답 품질을 직접 뜻하지 않습니다.


## 0. Colab 준비

먼저 **런타임 → 런타임 유형 변경 → GPU(T4) → 저장**을 선택하세요. 이전 설치로 패키지 상태가 꼬였으므로 가능하면 새 GPU 런타임에서 시작합니다.

아래 설치 셀은 CUDA GPU가 보이는지 먼저 확인하고, 현재 Torch 버전을 constraint로 고정해 GPTQModel 설치가 Torch를 CPU 빌드로 바꾸지 못하게 합니다. 설치 셀과 PyArrow 복구 셀을 모두 실행한 뒤 런타임을 다시 시작하고, 그 다음 import/GPU 확인 셀부터 실행하세요.

`ModuleNotFoundError`가 나거나 런타임을 새로 만들었다면 설치 셀부터 다시 실행합니다.


In [ ]:
import sys
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 먼저 선택하세요. CPU 런타임에는 GPTQModel을 설치하지 않습니다.")

torch_pin = torch.__version__
constraint_path = "/content/team22-torch-constraint.txt"
with open(constraint_path, "w") as f:
    f.write(f"torch=={torch_pin}\n")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-c", constraint_path, "gptqmodel==7.10.0", "datasets>=5.1.0"
])
print("설치 완료. 다음 PyArrow 복구 셀까지 실행한 뒤 런타임을 다시 시작하세요.")


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir", "--force-reinstall", "--no-deps",
    "pyarrow>=26.0.0"
])
print("PyArrow 재설치 완료. 이제 런타임을 다시 시작하세요.")


In [ ]:
import gc
import json
import math
import random
import sys
from pathlib import Path

import torch
import pandas as pd
import gptqmodel
import transformers
import datasets
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import BACKEND, GPTQModel, GPTQConfig, RTNConfig

assert torch.cuda.is_available(), "Colab 런타임을 GPU로 바꾸고 다시 실행하세요."
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPTQModel:", gptqmodel.__version__)
print("Datasets:", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. 실험 설정

평가 표본은 언어별 세 묶음, 각 묶음은 16개의 512-token 구간으로 구성합니다. GPTQ 보정 표본은 영어와 한국어 각각 128개, 최대 512 tokens입니다.

한국어 보정 표본과 한국어 평가 문서는 같은 Wikipedia 스냅샷에서 가져오되, 보정에 사용한 문서 ID는 평가 표본에서 제외합니다. 영어 보정은 영어 Wikipedia에서, 영어 평가는 WikiText-2 test에서 가져옵니다.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("/content/drive/MyDrive/Team22/pilot_qwen15b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WIKI_DATASET = "wikimedia/wikipedia"
WIKI_REVISION = "ad5752b5e625abfcdeefe5ae0ad2c3721c4b2619"
KOREAN_WIKI_CONFIG = "20231101.ko"
ENGLISH_WIKI_CONFIG = "20231101.en"
WIKITEXT_DATASET = "Salesforce/wikitext"
WIKITEXT_CONFIG = "wikitext-2-raw-v1"

SEED = 22
N_CALIBRATION = 128
CALIBRATION_LENGTH = 512
N_GROUPS = 3
SEQUENCES_PER_GROUP = 16
EVAL_SEQUENCE_LENGTH = 512
EVAL_TOKENS_PER_GROUP = SEQUENCES_PER_GROUP * EVAL_SEQUENCE_LENGTH

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

with open(OUTPUT_DIR / "runtime.json", "w", encoding="utf-8") as f:
    json.dump({"model_id": MODEL_ID, "wiki_revision": WIKI_REVISION, "seed": SEED,
               "torch": torch.__version__, "transformers": transformers.__version__,
               "gptqmodel": gptqmodel.__version__, "gpu": torch.cuda.get_device_name(0)},
              f, ensure_ascii=False, indent=2)


OUTPUT_DIR


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
print("Tokenizer:", tokenizer.name_or_path)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)


## 2. 고정된 보정·평가 표본 만들기

위키 문서를 seed로 섞어 보정 표본을 만듭니다. 보정에 쓴 문서 ID를 기억해 한국어 평가 표본에서 제외합니다. 각 평가 묶음은 서로 다른 문서에서 가져오고 토큰 수를 같게 맞춥니다. 표본 생성이 끝나면 텍스트를 JSON으로 저장해 런타임이 끊겨도 동일한 표본을 다시 쓸 수 있습니다.


In [ ]:
def get_text(row):
    text = row.get("text", "")
    return text.strip() if isinstance(text, str) else ""

def token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def load_wikipedia_stream(config, seed):
    ds = load_dataset(
        WIKI_DATASET, config, split="train", streaming=True, revision=WIKI_REVISION
    )
    return ds.shuffle(seed=seed, buffer_size=10_000)

def collect_calibration(lang, config):
    ds = load_wikipedia_stream(config, SEED + (0 if lang == "en" else 1))
    texts, ids = [], set()
    for row in ds:
        text = get_text(row)
        if not text:
            continue
        encoded = tokenizer(
            text, add_special_tokens=True, truncation=True,
            max_length=CALIBRATION_LENGTH, return_tensors="pt"
        )
        if encoded["input_ids"].shape[1] != CALIBRATION_LENGTH:
            continue
        texts.append(text)
        ids.add(str(row.get("id", "")))
        if len(texts) == N_CALIBRATION:
            break
    if len(texts) != N_CALIBRATION:
        raise RuntimeError(f"{lang}: 보정 문서를 {N_CALIBRATION}개 모으지 못했습니다.")
    return texts, ids

calib_en_texts, calib_en_ids = collect_calibration("en", ENGLISH_WIKI_CONFIG)
calib_ko_texts, calib_ko_ids = collect_calibration("ko", KOREAN_WIKI_CONFIG)
print("Calibration documents:", len(calib_en_texts), len(calib_ko_texts))


In [ ]:
def collect_korean_eval_groups(excluded_ids):
    ds = load_wikipedia_stream(KOREAN_WIKI_CONFIG, SEED + 100)
    groups = [[] for _ in range(N_GROUPS)]
    lengths = [0] * N_GROUPS
    group_index = 0
    for row in ds:
        if str(row.get("id", "")) in excluded_ids:
            continue
        text = get_text(row)
        if not text:
            continue
        groups[group_index].append(text)
        lengths[group_index] += token_count(text)
        if lengths[group_index] >= EVAL_TOKENS_PER_GROUP:
            if group_index == N_GROUPS - 1:
                break
            group_index += 1
    if any(length < EVAL_TOKENS_PER_GROUP for length in lengths):
        raise RuntimeError(f"한국어 평가 문서를 충분히 모으지 못했습니다: {lengths}")
    return groups

def collect_english_eval_groups():
    ds = load_dataset(WIKITEXT_DATASET, WIKITEXT_CONFIG, split="test")
    groups = [[] for _ in range(N_GROUPS)]
    for index, row in enumerate(ds):
        text = get_text(row)
        if text:
            groups[index % N_GROUPS].append(text)
    return groups

raw_eval_groups = {
    "ko": collect_korean_eval_groups(calib_ko_ids),
    "en": collect_english_eval_groups(),
}

def make_token_groups(text_groups):
    token_groups = []
    for texts in text_groups:
        ids = []
        for text in texts:
            ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
            if tokenizer.eos_token_id is not None:
                ids.append(tokenizer.eos_token_id)
            if len(ids) >= EVAL_TOKENS_PER_GROUP:
                break
        ids = ids[:EVAL_TOKENS_PER_GROUP]
        if len(ids) < EVAL_TOKENS_PER_GROUP:
            raise RuntimeError("평가 묶음 토큰 수가 부족합니다.")
        token_groups.append(ids)
    return token_groups

eval_token_groups = {
    lang: make_token_groups(groups) for lang, groups in raw_eval_groups.items()
}

with open(OUTPUT_DIR / "pilot_samples.json", "w", encoding="utf-8") as f:
    json.dump(
        {"calibration_en": calib_en_texts, "calibration_ko": calib_ko_texts,
         "eval_groups": raw_eval_groups},
        f, ensure_ascii=False
    )

print({lang: [len(group) for group in groups]
       for lang, groups in eval_token_groups.items()})


## 3. Perplexity 계산

각 문서는 512-token 구간으로 평가합니다. 구간별 평균 loss를 기록하고 `exp(loss)`로 Perplexity를 계산합니다. 세 묶음은 서로 다른 문서 집합이며, 모두 모델 조건 간에 동일하게 사용됩니다.


In [ ]:
def evaluate_ppl(model, token_groups, label):
    model.eval()
    rows = []
    device = torch.device("cuda:0")
    for lang, groups in token_groups.items():
        for group_idx, token_ids in enumerate(groups, start=1):
            total_nll = 0.0
            total_predicted_tokens = 0
            for start in range(0, len(token_ids), EVAL_SEQUENCE_LENGTH):
                chunk = token_ids[start:start + EVAL_SEQUENCE_LENGTH]
                if len(chunk) != EVAL_SEQUENCE_LENGTH:
                    continue
                input_ids = torch.tensor([chunk], dtype=torch.long, device=device)
                with torch.inference_mode():
                    output = model(input_ids=input_ids, labels=input_ids)
                n_predicted = input_ids.shape[1] - 1
                total_nll += float(output.loss.detach().float().cpu()) * n_predicted
                total_predicted_tokens += n_predicted
                del input_ids, output
            mean_nll = total_nll / total_predicted_tokens
            rows.append({
                "condition": label, "language": lang, "group": group_idx,
                "mean_nll": mean_nll, "ppl": math.exp(mean_nll),
                "predicted_tokens": total_predicted_tokens,
            })
    return rows

def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# FP16 기준선: 세 묶음의 PPL을 먼저 계산합니다.
fp16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0")
fp16_rows = evaluate_ppl(fp16_model, eval_token_groups, "FP16")
del fp16_model
clear_gpu_cache()

results = list(fp16_rows)
pd.DataFrame(results)


## 4. RTN 4bit 생성 및 평가

RTN은 보정 데이터 없이 nearest rounding 방식으로 양자화합니다. 새 모델 인스턴스에서 한 번만 양자화하고, 저장한 체크포인트를 불러와 평가합니다.


In [ ]:
def quantize_and_save(condition, quant_config, calibration_texts=None, backend=None):
    save_dir = OUTPUT_DIR / condition.lower().replace(" ", "_")
    weights_exist = any(save_dir.glob("*.safetensors")) or any(save_dir.glob("pytorch_model*.bin"))
    if (save_dir / "config.json").exists() and weights_exist:
        print(f"기존 체크포인트 사용: {save_dir}")
        return save_dir
    if save_dir.exists() and any(save_dir.iterdir()):
        raise RuntimeError(f"불완전한 기존 폴더입니다. 파일을 확인한 뒤 직접 정리하세요: {save_dir}")

    model = GPTQModel.load(
        MODEL_ID, quantize_config=quant_config,
        device=("cuda:0",), dtype=torch.float16
    )
    calibration = None
    if calibration_texts is not None:
        calibration = [
            tokenizer(text, truncation=True, max_length=CALIBRATION_LENGTH,
                      return_tensors="pt")
            for text in calibration_texts
        ]
    model.quantize(calibration, batch_size=1, backend=backend)
    model.save(str(save_dir))
    del model
    clear_gpu_cache()
    return save_dir

def load_quantized(path):
    return GPTQModel.load(
        str(path), device="cuda:0", dtype=torch.float16, backend=BACKEND.AUTO
    )

rtn_path = quantize_and_save(
    "RTN 4bit", RTNConfig(bits=4, group_size=128),
    calibration_texts=None, backend=BACKEND.AUTO
)
rtn_model = load_quantized(rtn_path)
results.extend(evaluate_ppl(rtn_model, eval_token_groups, "RTN 4bit"))
del rtn_model
clear_gpu_cache()
pd.DataFrame(results).tail(6)


## 5. 영어·한국어 보정 GPTQ 4bit 생성 및 평가

두 GPTQ 조건은 보정 언어만 다르게 하고 나머지 설정은 같습니다. 각 GPTQ는 FP16 원본에서 새로 시작합니다.


In [ ]:
for condition, calibration_texts in [
    ("GPTQ English 4bit", calib_en_texts),
    ("GPTQ Korean 4bit", calib_ko_texts),
]:
    path = quantize_and_save(
        condition, GPTQConfig(bits=4, group_size=128, sym=True, desc_act=False),
        calibration_texts=calibration_texts, backend=BACKEND.GPTQ_TORCH
    )
    quant_model = load_quantized(path)
    results.extend(evaluate_ppl(quant_model, eval_token_groups, condition))
    del quant_model
    clear_gpu_cache()
    print("완료:", condition)


## 6. 표와 예비 결과 정리

아래 표에서 `D`는 언어별 FP16 기준 Perplexity 증가율입니다. `D_gap_pp`가 양수면 해당 평가 묶음에서 한국어 저하율이 영어보다 높았다는 뜻입니다. 이는 예비 결과를 요약하는 값이며, 지금 여기서 1번/3번 선택을 자동으로 결정하지 않습니다.


In [ ]:
df = pd.DataFrame(results)
base = (df[df["condition"] == "FP16"]
        .set_index(["language", "group"])["ppl"])
df["D_percent"] = df.apply(
    lambda row: (row["ppl"] / base.loc[(row["language"], row["group"])] - 1) * 100
    if row["condition"] != "FP16" else 0.0, axis=1
)

wide_d = df.pivot_table(
    index=["condition", "group"], columns="language", values="D_percent"
).reset_index()
wide_d["D_gap_pp"] = wide_d["ko"] - wide_d["en"]

summary = (df.groupby(["condition", "language"], as_index=False)
           .agg(mean_ppl=("ppl", "mean"), sd_ppl=("ppl", "std"),
                mean_D_percent=("D_percent", "mean")))
gap_summary = (wide_d.groupby("condition", as_index=False)
               .agg(mean_D_gap_pp=("D_gap_pp", "mean"),
                    positive_gap_groups=("D_gap_pp", lambda x: int((x > 0).sum())),
                    n_groups=("D_gap_pp", "count")))

# 한국어 보정 GPTQ가 영어 보정 GPTQ보다 한국어 PPL을 낮췄는지 요약합니다.
ko_compare = df[df["language"] == "ko"].pivot(
    index="group", columns="condition", values="ppl"
)
ko_calibration_benefit = (
    (ko_compare["GPTQ English 4bit"] - ko_compare["GPTQ Korean 4bit"])
    / ko_compare["GPTQ English 4bit"] * 100
).rename("Korean_calibration_PPL_improvement_percent")

print("언어별 평균 결과")
display(summary)
print("한국어-영어 저하율 차이 (percentage points)")
display(gap_summary)
print("한국어 평가에서 한국어 보정이 영어 보정보다 개선한 비율 (양수면 개선)")
display(ko_calibration_benefit.to_frame())

df.to_csv(OUTPUT_DIR / "ppl_by_group.csv", index=False)
summary.to_csv(OUTPUT_DIR / "ppl_summary.csv", index=False)
gap_summary.to_csv(OUTPUT_DIR / "language_gap_summary.csv", index=False)
ko_calibration_benefit.to_csv(OUTPUT_DIR / "korean_calibration_effect.csv")


## 7. 결과를 보고 확인할 것

- **언어별 양자화 영향:** RTN/GPTQ마다 `D_percent`와 `D_gap_pp`를 확인합니다. 세 평가 묶음에서 방향이 같은지 함께 봅니다.
- **보정 언어 영향:** 한국어 평가에서 `GPTQ Korean 4bit`의 PPL이 `GPTQ English 4bit`보다 낮은지 봅니다. 반대 언어 평가도 같이 확인합니다.
- **변동성:** 묶음별 PPL과 저하율을 나란히 보고, 평균만으로 결론 내리지 않습니다.
- **주제 선택:** 결과를 팀에서 검토한 뒤 1번과 3번 중 하나를 고릅니다. 3번을 택하면 그때 부품 민감도와 혼합 정밀도 후보를 설계합니다.
- **해석 한계:** 서로 다른 말뭉치의 절대 PPL을 한국어-영어 간 직접 비교하지 않습니다. 이 예비 실험은 퍼플렉시티 변화의 방향을 보는 것이며, KoBEST 정확도나 실제 대화 품질을 대신하지 않습니다.

결과 파일은 Google Drive의 `/content/drive/MyDrive/Team22/pilot_qwen15b`에 저장됩니다. 이 폴더는 Drive 연결 후 코드가 자동으로 만듭니다.
